# 07 - Entrenamiento Práctico de Redes Neuronales

**AI sin humo** - Notas personales para entender deep learning desde cero.

En el notebook anterior vimos backpropagation: cómo calcular gradientes para todos los parámetros de la red. Genial. Pero en la práctica, hacer gradient descent "puro" con todos los datos es **lento, inestable y se queda atascado en mínimos malos**. Este notebook es sobre las técnicas que hacen que el entrenamiento **realmente funcione** en la práctica.

Vamos a cubrir los problemas más comunes del entrenamiento y las soluciones que usa todo el mundo: SGD, momentum, Adam, learning rate schedules, inicialización inteligente de parámetros, normalización (BatchNorm, LayerNorm) y residual connections. Esto es lo que separa a alguien que entiende la teoría de alguien que puede entrenar modelos de verdad.

---

## Contenido

1. [Tres grandes problemas del entrenamiento](#problemas)
2. [SGD (Stochastic Gradient Descent)](#sgd)
3. [Batches y Epochs](#batches-epochs)
4. [Learning rate schedule](#lr-schedule)
5. [Momentum](#momentum)
6. [Adam Optimizer](#adam)
7. [Inicialización de parámetros](#inicializacion)
8. [BatchNorm](#batchnorm)
9. [LayerNorm](#layernorm)
10. [Residual connections](#residual)
11. [Resumen](#resumen)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn

---

<a id='problemas'></a>
## 1. Tres grandes problemas del entrenamiento

Hasta ahora parece todo muy lindo: calculamos gradientes, actualizamos parámetros, y la loss baja. ¿Qué puede salir mal?

**Casi todo.**

En la práctica, entrenar redes neuronales es difícil. Hay tres problemas fundamentales que aparecen siempre:

### Problema 1: Inestabilidad (la loss oscila como loca)

Usás gradient descent con un learning rate razonable y la loss sube y baja sin parar. A veces explota directamente a infinito. Esto pasa porque la **superficie de loss no es una montaña suavecita**: tiene valles estrechos, curvas pronunciadas y zonas donde el gradiente es enorme en una dirección y chiquito en otra.

Si el learning rate es muy grande para las zonas pronunciadas, el paso es tan largo que te pasás del mínimo y terminás más lejos de donde empezaste. Es como intentar bajar una montaña con botas de 7 leguas: saltás de un lado al otro del valle sin nunca llegar al fondo.

### Problema 2: Convergencia lenta (la loss baja pero a paso de tortuga)

Lo opuesto: el learning rate es chico, no explota, pero tarda una eternidad en converger. O peor: la loss baja rapidísimo en una dirección (donde el gradiente es grande) pero apenas se mueve en otra (donde el gradiente es chiquito). Esto genera trayectorias en zigzag que son super ineficientes.

Pensá en un valle largo y angosto: el gradiente te empuja fuerte hacia las paredes del valle (dirección transversal) pero apenas te mueve a lo largo del valle (dirección del mínimo). Resultado: zigzagueo infinito.

### Problema 3: Mínimos malos (local minima y saddle points)

La superficie de loss de una red neuronal es **no convexa**. Esto es fundamental y hay que entenderlo bien.

Una función convexa tiene un único mínimo: no importa de dónde arranques, gradient descent te lleva ahí. Pero la loss de una red neuronal tiene:

- **Mínimos locales**: puntos donde la loss es mínima "localmente" pero no globalmente. El gradiente es cero, así que gradient descent se detiene, pero hay puntos mejores en otro lado.
- **Saddle points** (puntos de silla): puntos donde el gradiente también es cero, pero no son mínimos ni máximos — la función sube en una dirección y baja en otra. En espacios de alta dimensión, estos son **mucho más comunes** que los mínimos locales.
- **Mesetas** (plateaus): regiones planas donde el gradiente es casi cero. El entrenamiento se estanca porque no hay señal de en qué dirección ir.

![Local minima y saddle points en la superficie de loss](../ai_notas/AI%20notas/image%2047.png)

### ¿Por qué la superficie es no convexa?

Porque las redes neuronales son **composiciones de funciones no lineales**. Tenés capas lineales seguidas de activaciones no lineales (ReLU, sigmoid, etc.), compuestas una tras otra. El resultado es una función mega-compleja con una superficie de loss llena de montañas, valles, pasos de montaña y mesetas.

De hecho, un resultado conocido es que en redes grandes, la mayoría de los puntos críticos (gradiente = 0) son **saddle points, no mínimos locales**. La buena noticia es que en la práctica, con las técnicas que vamos a ver, podemos entrenar redes enormes sin problemas.

### Entonces, ¿qué hacemos?

Todas las técnicas de este notebook atacan estos tres problemas:

| Técnica | Ataca |
|:--------|:------|
| **SGD** | Ruido estocástico ayuda a escapar saddle points |
| **Learning rate schedule** | Estabilidad + convergencia |
| **Momentum** | Convergencia lenta + zigzagueo |
| **Adam** | Los tres (adaptativo por parámetro) |
| **Inicialización** | Estabilidad desde el arranque |
| **BatchNorm / LayerNorm** | Estabilidad + convergencia rápida |
| **Residual connections** | Permite redes profundas sin degradación |

In [ ]:
# Let's visualize these problems on a 2D loss surface
# Rosenbrock-like function: classic example of a hard optimization landscape

def hard_loss_surface(x, y):
    """Non-convex surface with local minima, saddle points, and narrow valleys."""
    return (1 - x)**2 + 10 * (y - x**2)**2 + 0.5 * np.sin(5 * x) * np.sin(5 * y)

x_range = np.linspace(-2, 2, 300)
y_range = np.linspace(-1, 3, 300)
X, Y = np.meshgrid(x_range, y_range)
Z = hard_loss_surface(X, Y)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Problem 1: Instability (large LR)
ax = axes[0]
ax.contour(X, Y, Z, levels=50, cmap='viridis', alpha=0.7)
# Simulate oscillating path with large LR
path_x = [-0.5, 0.8, -0.3, 1.2, -0.1, 0.9, 0.2, 1.1]
path_y = [2.0, 0.5, 2.5, 0.2, 1.8, 0.8, 1.5, 1.0]
ax.plot(path_x, path_y, 'ro-', markersize=5, linewidth=1.5, alpha=0.8)
ax.set_title('Problema 1: Inestabilidad\n(LR muy grande → oscila)', fontsize=11)
ax.set_xlabel('θ₁'); ax.set_ylabel('θ₂')

# Problem 2: Slow convergence (small LR)
ax = axes[1]
ax.contour(X, Y, Z, levels=50, cmap='viridis', alpha=0.7)
# Simulate slow zigzag path
t = np.linspace(0, 1, 30)
path_x = -1.5 + 2.5 * t + 0.3 * np.sin(15 * t) * (1 - t)
path_y = 2.5 - 1.5 * t + 0.2 * np.cos(15 * t) * (1 - t)
ax.plot(path_x, path_y, 'bo-', markersize=3, linewidth=1, alpha=0.8)
ax.set_title('Problema 2: Convergencia lenta\n(zigzagueo ineficiente)', fontsize=11)
ax.set_xlabel('θ₁'); ax.set_ylabel('θ₂')

# Problem 3: Bad minima
ax = axes[2]
ax.contour(X, Y, Z, levels=50, cmap='viridis', alpha=0.7)
# Point stuck at a saddle/local min
ax.plot([-0.5], [0.5], 'rs', markersize=12, label='Mínimo local')
ax.plot([1.0], [1.0], 'g*', markersize=15, label='Mínimo global')
ax.plot([0.0], [1.5], 'k^', markersize=10, label='Saddle point')
ax.set_title('Problema 3: Mínimos malos\n(local minima y saddle points)', fontsize=11)
ax.set_xlabel('θ₁'); ax.set_ylabel('θ₂')
ax.legend(fontsize=9)

plt.tight_layout()
plt.show()

---

<a id='sgd'></a>
## 2. SGD (Stochastic Gradient Descent)

El gradient descent "vanilla" calcula el gradiente usando **todos los datos** del dataset:

$$\theta \leftarrow \theta - \eta \cdot \nabla_{\theta} \frac{1}{N} \sum_{i=1}^{N} L(f(x_i; \theta), y_i)$$

Esto tiene dos problemas enormes:

1. **Es carísimo**: si tenés un millón de datos, necesitás hacer un forward y backward pass por CADA dato antes de dar UN paso. Una sola actualización de parámetros requiere procesar todo el dataset.

2. **Es determinístico**: cada vez que estás en el mismo punto, calculás exactamente el mismo gradiente. Si estás en un saddle point o un mínimo local, te quedás ahí para siempre.

### La idea de SGD

En vez de calcular el gradiente con TODO el dataset, usamos un **subconjunto aleatorio** (un minibatch) en cada paso:

$$\theta \leftarrow \theta - \eta \cdot \nabla_{\theta} \frac{1}{|B|} \sum_{i \in B} L(f(x_i; \theta), y_i)$$

Donde $B$ es un minibatch aleatorio de, digamos, 32 o 64 ejemplos.

### ¿Por qué funciona?

La clave es que el gradiente calculado con un minibatch es un **estimador ruidoso** del gradiente real. En promedio, apunta en la misma dirección que el gradiente completo, pero tiene ruido.

Y ese ruido es **bueno**:

- **Escapar mínimos malos**: el ruido puede empujar al optimizador fuera de un mínimo local poco profundo o un saddle point. Cada minibatch te da una dirección ligeramente diferente.
- **Eficiencia**: en vez de esperar a procesar todo el dataset para dar un paso, damos muchos pasos chiquitos. 1000 pasos con batches de 32 ≈ cubrir el dataset si tiene ~32000 ejemplos, pero con 1000 actualizaciones en vez de 1.
- **Generalización**: hay evidencia de que el ruido de SGD actúa como regularizador implícito, ayudando al modelo a encontrar mínimos "anchos" que generalizan mejor.

### El tradeoff del batch size

| Batch size | Ruido | Cómputo por paso | Pasos para converger |
|:-----------|:------|:------------------|:---------------------|
| 1 (SGD puro) | Máximo | Mínimo | Muchos, muy ruidoso |
| 32-128 (típico) | Moderado | Moderado | Buen balance |
| Todo el dataset | Cero (determinístico) | Máximo | Pocos pero lentos |

En la práctica, batch sizes de **32 a 256** son los más comunes. Más grande que eso empieza a perder la ventaja del ruido y necesitás más memoria GPU.

In [ ]:
# Compare full-batch GD vs SGD on a simple problem
np.random.seed(42)

# Generate synthetic data: y = 3x + 1 + noise
N = 200
x_data = np.random.randn(N, 1)
y_data = 3 * x_data + 1 + 0.5 * np.random.randn(N, 1)

def compute_loss_and_grad(w, b, x, y):
    """MSE loss and gradients for y_hat = w*x + b."""
    y_hat = w * x + b
    loss = np.mean((y - y_hat) ** 2)
    dw = -2 * np.mean((y - y_hat) * x)
    db = -2 * np.mean(y - y_hat)
    return loss, dw, db

# --- Full-batch gradient descent ---
w_gd, b_gd = 0.0, 0.0
lr = 0.05
losses_gd = []
path_gd = [(w_gd, b_gd)]

for step in range(100):
    loss, dw, db = compute_loss_and_grad(w_gd, b_gd, x_data, y_data)
    losses_gd.append(loss)
    w_gd -= lr * dw
    b_gd -= lr * db
    path_gd.append((w_gd, b_gd))

# --- SGD with minibatches ---
w_sgd, b_sgd = 0.0, 0.0
batch_size = 16
losses_sgd = []
path_sgd = [(w_sgd, b_sgd)]

for step in range(100):
    # Random minibatch
    idx = np.random.choice(N, batch_size, replace=False)
    x_batch = x_data[idx]
    y_batch = y_data[idx]
    
    # Compute loss on full dataset (for comparison)
    full_loss, _, _ = compute_loss_and_grad(w_sgd, b_sgd, x_data, y_data)
    losses_sgd.append(full_loss)
    
    # Compute gradient on minibatch only
    _, dw, db = compute_loss_and_grad(w_sgd, b_sgd, x_batch, y_batch)
    w_sgd -= lr * dw
    b_sgd -= lr * db
    path_sgd.append((w_sgd, b_sgd))

# --- Plot comparison ---
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Loss curves
ax = axes[0]
ax.plot(losses_gd, 'b-', linewidth=2, label='Full-batch GD', alpha=0.8)
ax.plot(losses_sgd, 'r-', linewidth=1.5, label=f'SGD (batch={batch_size})', alpha=0.8)
ax.set_xlabel('Paso', fontsize=11)
ax.set_ylabel('Loss (MSE)', fontsize=11)
ax.set_title('Loss: Full-batch GD vs SGD', fontsize=12)
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)

# Paths in parameter space
ax = axes[1]
path_gd = np.array(path_gd)
path_sgd = np.array(path_sgd)
ax.plot(path_gd[:, 0], path_gd[:, 1], 'b.-', markersize=3, linewidth=1.5,
        label='Full-batch GD', alpha=0.7)
ax.plot(path_sgd[:, 0], path_sgd[:, 1], 'r.-', markersize=3, linewidth=1,
        label='SGD', alpha=0.7)
ax.plot(3.0, 1.0, 'g*', markersize=15, label='Óptimo (w=3, b=1)')
ax.plot(0.0, 0.0, 'ko', markersize=8, label='Inicio')
ax.set_xlabel('w', fontsize=11)
ax.set_ylabel('b', fontsize=11)
ax.set_title('Trayectoria en espacio de parámetros', fontsize=12)
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"Full-batch GD final: w={path_gd[-1, 0]:.4f}, b={path_gd[-1, 1]:.4f}")
print(f"SGD final:           w={path_sgd[-1, 0]:.4f}, b={path_sgd[-1, 1]:.4f}")
print(f"\nSGD llega a un resultado similar pero con un camino más ruidoso.")
print(f"Ese ruido es BUENO: ayuda a explorar y escapar mínimos malos.")

Fijate cómo SGD tiene un camino mucho más "ruidoso" que full-batch GD, pero llega a un resultado igual de bueno. En problemas más complejos (redes profundas, superficies no convexas), ese ruido es lo que marca la diferencia entre quedarse atascado y encontrar una buena solución.

---

<a id='batches-epochs'></a>
## 3. Batches y Epochs

Antes de seguir, aclaremos la terminología que todo el mundo usa:

### Minibatch

Un **minibatch** es un subconjunto del dataset, tomado **sin reemplazo** dentro de cada epoch. Si tenés 1000 datos y un batch size de 100, en cada epoch procesás 10 minibatches, y cada dato aparece exactamente una vez.

Esto es diferente de sampling con reemplazo (donde un dato podría aparecer dos veces en un epoch y otro ninguna). Sin reemplazo reduce la varianza del estimador del gradiente.

### Epoch

Un **epoch** es un pase completo por todo el dataset. Después de un epoch, ya viste todos los datos una vez. Antes de empezar el siguiente epoch, mezclás los datos de nuevo (shuffle) y formás nuevos minibatches.

```
Dataset: [d1, d2, d3, d4, d5, d6, d7, d8, d9]   (9 datos, batch_size=3)

Epoch 1:   Shuffle → [d5, d2, d8, d1, d7, d3, d9, d4, d6]
           Batch 1: [d5, d2, d8]  → calcular gradiente → actualizar θ
           Batch 2: [d1, d7, d3]  → calcular gradiente → actualizar θ
           Batch 3: [d9, d4, d6]  → calcular gradiente → actualizar θ

Epoch 2:   Shuffle → [d3, d6, d1, d8, d4, d9, d2, d5, d7]
           Batch 1: [d3, d6, d1]  → calcular gradiente → actualizar θ
           ...y así
```

### Una forma de pensar SGD que es muy útil

Podés pensar que en cada paso de SGD, estás optimizando una **loss function diferente**. La loss function depende de dos cosas:

1. **El modelo** (los parámetros actuales)
2. **Los datos** (el minibatch actual)

Si cambiás el minibatch, cambiás la loss function. Entonces, en cada iteración estás calculando el gradiente de una función ligeramente diferente.

$$L_1(\theta) = \frac{1}{|B_1|} \sum_{i \in B_1} \ell(f(x_i; \theta), y_i) \quad \text{(loss del batch 1)}$$

$$L_2(\theta) = \frac{1}{|B_2|} \sum_{i \in B_2} \ell(f(x_i; \theta), y_i) \quad \text{(loss del batch 2)}$$

Cada una de estas funciones tiene su propia superficie, sus propios mínimos y gradientes. El modelo está optimizando una secuencia de problemas ligeramente diferentes. Pero en promedio, todas estas funciones comparten el mismo mínimo (el de la loss verdadera sobre todo el dataset).

Esta perspectiva explica por qué SGD puede escapar mínimos locales: un mínimo local de $L_1$ puede no ser un mínimo local de $L_2$, así que al cambiar de batch, el gradiente te saca de ahí.

In [ ]:
# Demonstrate: each minibatch defines a slightly different loss function
np.random.seed(42)

# Generate data
N = 200
x_data = np.random.randn(N, 1)
y_data = 2.5 * x_data + 0.8 + 0.5 * np.random.randn(N, 1)

# Compute loss surface for different batches
w_range = np.linspace(0, 5, 100)
b_range = np.linspace(-2, 3, 100)
W, B = np.meshgrid(w_range, b_range)

fig, axes = plt.subplots(1, 4, figsize=(18, 4))

batch_size = 16
np.random.shuffle(x_data)  # shuffle indices via data

for i, ax in enumerate(axes):
    if i < 3:
        # Different minibatches
        idx = np.random.choice(N, batch_size, replace=False)
        x_b = x_data[idx]
        y_b = y_data[idx]
        title = f'Batch {i+1} ({batch_size} datos)'
    else:
        # Full dataset
        x_b = x_data
        y_b = y_data
        title = f'Dataset completo ({N} datos)'
    
    # Compute loss surface
    L = np.zeros_like(W)
    for j in range(len(x_b)):
        L += (y_b[j] - (W * x_b[j] + B)) ** 2
    L /= len(x_b)
    
    # Find minimum
    min_idx = np.unravel_index(L.argmin(), L.shape)
    
    ax.contour(W, B, L, levels=20, cmap='viridis', alpha=0.7)
    ax.plot(W[min_idx], B[min_idx], 'r*', markersize=12)
    ax.set_xlabel('w', fontsize=10)
    ax.set_ylabel('b', fontsize=10)
    ax.set_title(title, fontsize=10)

plt.suptitle('Cada minibatch define una loss function ligeramente diferente\n'
             '(las estrellas rojas son los mínimos — ¡se mueven!)',
             fontsize=12, y=1.05)
plt.tight_layout()
plt.show()

Fijate cómo el mínimo de cada batch está en un lugar ligeramente diferente. El mínimo del dataset completo (último gráfico) es el "verdadero" mínimo, y los de cada batch son estimaciones ruidosas. Pero en promedio, todos apuntan al mismo lugar.

---

<a id='lr-schedule'></a>
## 4. Learning rate schedule

El learning rate ($\eta$) es probablemente el hiperparámetro más importante de todo el entrenamiento. Y resulta que usar un learning rate **fijo** durante todo el entrenamiento es casi siempre subóptimo.

### La intuición

- **Al principio** del entrenamiento: estás lejos del mínimo, querés explorar rápido y dar pasos grandes. Un LR grande te permite cubrir mucho terreno.
- **Al final** del entrenamiento: estás cerca del mínimo, necesitás pasos chiquitos para no pasarte de largo. Un LR grande te haría oscilar alrededor del mínimo sin converger.

Entonces la solución es obvia: **empezar con un LR grande e ir achicándolo** a medida que avanza el entrenamiento.

### ¿Qué pasa si NO usás schedule?

- **LR fijo grande**: explora bien al principio, pero no converge al final (oscila)
- **LR fijo chico**: converge al final, pero es lentísimo al principio y puede quedarse en un mínimo malo
- **LR que decrece**: lo mejor de los dos mundos

### Schedules más comunes

**Step decay**: bajar el LR por un factor cada N epochs
$$\eta_t = \eta_0 \cdot \gamma^{\lfloor t / T \rfloor}$$

**Cosine annealing**: decaimiento suave con forma de coseno
$$\eta_t = \eta_{min} + \frac{1}{2}(\eta_{max} - \eta_{min})(1 + \cos(\frac{t \cdot \pi}{T}))$$

**Warmup + decay**: empezar con LR bajo, subir linealmente (warmup), y después bajar. Esto estabiliza el comienzo del entrenamiento cuando los gradientes iniciales son impredecibles.

$$\eta_t = \begin{cases} \eta_{max} \cdot \frac{t}{T_{warmup}} & \text{si } t < T_{warmup} \\ \eta_{min} + \frac{1}{2}(\eta_{max} - \eta_{min})(1 + \cos(\frac{(t - T_{warmup}) \cdot \pi}{T - T_{warmup}})) & \text{si } t \geq T_{warmup} \end{cases}$$

In [ ]:
# Visualize different learning rate schedules

total_steps = 1000
lr_max = 0.01
lr_min = 1e-5

steps = np.arange(total_steps)

# 1. Constant
lr_constant = np.full(total_steps, lr_max)

# 2. Step decay (halve every 250 steps)
lr_step = lr_max * (0.5 ** (steps // 250))

# 3. Cosine annealing
lr_cosine = lr_min + 0.5 * (lr_max - lr_min) * (1 + np.cos(steps * np.pi / total_steps))

# 4. Warmup + cosine decay
warmup_steps = 100
lr_warmup_cosine = np.zeros(total_steps)
for t in range(total_steps):
    if t < warmup_steps:
        lr_warmup_cosine[t] = lr_max * t / warmup_steps
    else:
        progress = (t - warmup_steps) / (total_steps - warmup_steps)
        lr_warmup_cosine[t] = lr_min + 0.5 * (lr_max - lr_min) * (1 + np.cos(progress * np.pi))

fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(steps, lr_constant, '--', label='Constante', alpha=0.7, linewidth=2)
ax.plot(steps, lr_step, label='Step decay (×0.5 cada 250)', linewidth=2)
ax.plot(steps, lr_cosine, label='Cosine annealing', linewidth=2)
ax.plot(steps, lr_warmup_cosine, label='Warmup + cosine', linewidth=2)
ax.set_xlabel('Paso', fontsize=11)
ax.set_ylabel('Learning rate', fontsize=11)
ax.set_title('Diferentes learning rate schedules', fontsize=13)
ax.legend(fontsize=10)
ax.set_yscale('log')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Compare fixed LR vs cosine schedule on a harder optimization problem
np.random.seed(42)

# Non-convex 1D function with local minima
def f(x):
    return x**2 + 3 * np.sin(2 * x)

def grad_f(x):
    return 2 * x + 6 * np.cos(2 * x)

n_steps = 200

# Fixed LR (too large)
x_fixed_large = 4.0
path_fixed_large = [x_fixed_large]
lr_fixed_large = 0.15
for _ in range(n_steps):
    x_fixed_large -= lr_fixed_large * grad_f(x_fixed_large)
    path_fixed_large.append(x_fixed_large)

# Fixed LR (small, safe)
x_fixed_small = 4.0
path_fixed_small = [x_fixed_small]
lr_fixed_small = 0.01
for _ in range(n_steps):
    x_fixed_small -= lr_fixed_small * grad_f(x_fixed_small)
    path_fixed_small.append(x_fixed_small)

# Cosine schedule
x_cosine = 4.0
path_cosine = [x_cosine]
for t in range(n_steps):
    lr = 0.001 + 0.5 * (0.15 - 0.001) * (1 + np.cos(t * np.pi / n_steps))
    x_cosine -= lr * grad_f(x_cosine)
    path_cosine.append(x_cosine)

# Plot
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

x_plot = np.linspace(-5, 5, 500)
ax = axes[0]
ax.plot(x_plot, f(x_plot), 'k-', linewidth=2, alpha=0.5)
ax.plot(path_fixed_large[:50], [f(x) for x in path_fixed_large[:50]], 'r.-',
        markersize=4, alpha=0.7, label=f'LR fijo={lr_fixed_large} (oscila)')
ax.plot(path_fixed_small, [f(x) for x in path_fixed_small], 'b.-',
        markersize=2, alpha=0.7, label=f'LR fijo={lr_fixed_small} (lento)')
ax.plot(path_cosine, [f(x) for x in path_cosine], 'g.-',
        markersize=2, alpha=0.7, label='Cosine schedule')
ax.set_xlabel('x', fontsize=11)
ax.set_ylabel('f(x)', fontsize=11)
ax.set_title('Trayectorias de optimización', fontsize=12)
ax.legend(fontsize=9)
ax.set_ylim(-5, 25)
ax.grid(True, alpha=0.3)

ax = axes[1]
losses_large = [f(x) for x in path_fixed_large[:50]]
losses_small = [f(x) for x in path_fixed_small]
losses_cosine = [f(x) for x in path_cosine]
ax.plot(losses_large, 'r-', label=f'LR fijo={lr_fixed_large}', alpha=0.7)
ax.plot(losses_small, 'b-', label=f'LR fijo={lr_fixed_small}', alpha=0.7)
ax.plot(losses_cosine, 'g-', label='Cosine schedule', alpha=0.7, linewidth=2)
ax.set_xlabel('Paso', fontsize=11)
ax.set_ylabel('f(x)', fontsize=11)
ax.set_title('Loss durante optimización', fontsize=12)
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"LR fijo grande: x final = {path_fixed_large[-1]:.4f}, f(x) = {f(path_fixed_large[-1]):.4f}")
print(f"LR fijo chico:  x final = {path_fixed_small[-1]:.4f}, f(x) = {f(path_fixed_small[-1]):.4f}")
print(f"Cosine schedule: x final = {path_cosine[-1]:.4f}, f(x) = {f(path_cosine[-1]):.4f}")

El schedule de coseno combina lo mejor de ambos mundos: empieza rápido para explorar y termina lento para converger. Sin schedule, tenés que elegir entre velocidad (LR grande, riesgo de inestabilidad) o seguridad (LR chico, lento).

---

<a id='momentum'></a>
## 5. Momentum

Momentum es una de las mejoras más simples y efectivas sobre SGD básico. La idea es **darle memoria al optimizador**: en vez de solo mirar el gradiente actual, también recordamos los gradientes anteriores.

### El problema que resuelve

Recordá el problema del zigzagueo: en un valle alargado, SGD rebota de pared a pared sin avanzar en la dirección del mínimo. Cada gradiente apunta hacia la pared del valle (transversal), no a lo largo del valle.

### La idea física

Imaginá una bolita rodando por la superficie de loss. Si la bolita tiene masa (inercia), no cambia de dirección instantáneamente con cada gradiente nuevo. Tiende a seguir moviéndose en la dirección que ya llevaba. Si los gradientes están alineados (apuntan consistentemente en la misma dirección), la bolita se **acelera**. Si los gradientes cambian de dirección constantemente (zigzagueo), se **cancelan** entre sí.

### La fórmula

SGD con momentum mantiene una **velocidad** $v$ que es un promedio exponencial de los gradientes pasados:

$$v_t = \beta \cdot v_{t-1} + (1 - \beta) \cdot \nabla L(\theta_{t-1})$$
$$\theta_t = \theta_{t-1} - \eta \cdot v_t$$

Donde $\beta$ es el coeficiente de momentum (típicamente 0.9).

### ¿Qué hace el momentum exactamente?

Es como una **memoria exponencial** de los gradientes pasados, que se va olvidando de a poco:

$$v_t = (1-\beta) \cdot g_t + \beta(1-\beta) \cdot g_{t-1} + \beta^2(1-\beta) \cdot g_{t-2} + ...$$

Los gradientes recientes pesan más, los viejos pesan menos (porque $\beta < 1$ y se multiplica por sí mismo varias veces). Con $\beta = 0.9$:

- El gradiente de hace 1 paso contribuye con peso $0.9 \times 0.1 = 0.09$
- El de hace 10 pasos: $0.9^{10} \times 0.1 \approx 0.035$
- El de hace 50 pasos: $0.9^{50} \times 0.1 \approx 0.0005$ (ya casi nada)

### El efecto sobre el learning rate efectivo

Si los gradientes están **consistentemente alineados** (misma dirección):

$$v \approx \frac{1}{1 - \beta} \cdot g = \frac{g}{1 - 0.9} = 10g$$

¡El learning rate efectivo se multiplica por $\frac{1}{1-\beta}$! Con $\beta = 0.9$, es como multiplicar el LR por 10 en las direcciones consistentes.

Si los gradientes **cambian de signo** constantemente (zigzagueo), el término $\beta \cdot v_{t-1}$ y $g_t$ se cancelan, resultando en pasos mucho más chicos. Exactamente lo que queremos.

In [ ]:
# Compare SGD vs SGD+Momentum on a valley-shaped loss surface
np.random.seed(42)

# Elongated valley (different curvatures in x and y)
def valley_loss(x, y):
    return 0.1 * (x - 3)**2 + 5 * (y - 1)**2

def valley_grad(x, y):
    return np.array([0.2 * (x - 3), 10 * (y - 1)])

# --- SGD without momentum ---
pos_sgd = np.array([-2.0, 4.0])
path_sgd = [pos_sgd.copy()]
lr = 0.08

for _ in range(80):
    g = valley_grad(*pos_sgd)
    pos_sgd -= lr * g
    path_sgd.append(pos_sgd.copy())

# --- SGD with momentum ---
pos_mom = np.array([-2.0, 4.0])
path_mom = [pos_mom.copy()]
v = np.zeros(2)
beta = 0.9

for _ in range(80):
    g = valley_grad(*pos_mom)
    v = beta * v + (1 - beta) * g
    pos_mom -= lr * v
    path_mom.append(pos_mom.copy())

# --- SGD with high momentum ---
pos_hmom = np.array([-2.0, 4.0])
path_hmom = [pos_hmom.copy()]
v_h = np.zeros(2)
beta_h = 0.95

for _ in range(80):
    g = valley_grad(*pos_hmom)
    v_h = beta_h * v_h + (1 - beta_h) * g
    pos_hmom -= lr * v_h
    path_hmom.append(pos_hmom.copy())

# Plot
path_sgd = np.array(path_sgd)
path_mom = np.array(path_mom)
path_hmom = np.array(path_hmom)

x_r = np.linspace(-4, 7, 200)
y_r = np.linspace(-1, 5, 200)
X, Y = np.meshgrid(x_r, y_r)
Z = valley_loss(X, Y)

fig, ax = plt.subplots(figsize=(12, 7))
ax.contour(X, Y, Z, levels=30, cmap='viridis', alpha=0.6)
ax.plot(path_sgd[:, 0], path_sgd[:, 1], 'r.-', markersize=4, linewidth=1.5,
        alpha=0.8, label='SGD sin momentum')
ax.plot(path_mom[:, 0], path_mom[:, 1], 'b.-', markersize=4, linewidth=1.5,
        alpha=0.8, label=f'SGD + momentum (β={beta})')
ax.plot(path_hmom[:, 0], path_hmom[:, 1], 'g.-', markersize=4, linewidth=1.5,
        alpha=0.8, label=f'SGD + momentum (β={beta_h})')
ax.plot(3, 1, 'k*', markersize=15, label='Mínimo')
ax.plot(-2, 4, 'ko', markersize=8, label='Inicio')
ax.set_xlabel('θ₁', fontsize=12)
ax.set_ylabel('θ₂', fontsize=12)
ax.set_title('SGD vs Momentum en un valle alargado', fontsize=13)
ax.legend(fontsize=10, loc='upper right')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# Show loss over time
losses_sgd = [valley_loss(*p) for p in path_sgd]
losses_mom = [valley_loss(*p) for p in path_mom]
losses_hmom = [valley_loss(*p) for p in path_hmom]

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(losses_sgd, 'r-', label='SGD', alpha=0.7)
ax.plot(losses_mom, 'b-', label=f'Momentum β={beta}', alpha=0.7)
ax.plot(losses_hmom, 'g-', label=f'Momentum β={beta_h}', alpha=0.7)
ax.set_xlabel('Paso', fontsize=11)
ax.set_ylabel('Loss', fontsize=11)
ax.set_title('Convergencia: SGD vs Momentum', fontsize=12)
ax.legend(fontsize=10)
ax.set_yscale('log')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("SGD zigzaguea en el valle. Momentum \"recuerda\" la dirección consistente")
print("y cancela las oscilaciones transversales.")
print(f"\nPaso final SGD:            loss = {losses_sgd[-1]:.6f}")
print(f"Paso final Momentum 0.9:   loss = {losses_mom[-1]:.6f}")
print(f"Paso final Momentum 0.95:  loss = {losses_hmom[-1]:.6f}")

Fijate la diferencia dramática: SGD sin momentum zigzaguea constantemente (los gradientes transversales al valle se cancelan en momentum, pero en SGD puro se siguen de forma ciega). Con momentum, el optimizador "plancha" las oscilaciones y se mueve mucho más eficientemente a lo largo del valle.

Un valor típico de $\beta$ es **0.9**, que funciona bien en la mayoría de los casos. Valores más altos como 0.95 o 0.99 dan más "inercia" pero pueden hacer que el optimizador se pase del mínimo.

---

<a id='adam'></a>
## 6. Adam Optimizer

**Adam** (Adaptive Moment Estimation) es el optimizer más usado en deep learning. Combina dos ideas poderosas:

1. **Momentum**: promediar gradientes pasados para suavizar la dirección
2. **Normalización adaptativa**: cada parámetro tiene su propio learning rate efectivo, basado en la magnitud histórica de sus gradientes

### El problema que resuelve

Pensá en una red con millones de parámetros. Algunos parámetros reciben gradientes grandes (están en capas que se actualizan mucho) y otros reciben gradientes chiquitos (capas profundas, features raros). Con un LR fijo:

- Si el LR es bueno para los gradientes grandes, es demasiado chico para los chiquitos → esos parámetros aprenden lento
- Si el LR es bueno para los gradientes chiquitos, es demasiado grande para los grandes → inestabilidad

Adam dice: **que cada parámetro tenga su propio paso**, adaptado a la escala de sus gradientes.

### Las fórmulas

Adam mantiene dos promedios exponenciales (momentos):

**Primer momento** (media de gradientes = momentum):
$$m_t = \beta_1 \cdot m_{t-1} + (1 - \beta_1) \cdot g_t$$

**Segundo momento** (media de gradientes al cuadrado = varianza):
$$v_t = \beta_2 \cdot v_{t-1} + (1 - \beta_2) \cdot g_t^2$$

**Corrección de sesgo** (porque empiezan en 0):
$$\hat{m}_t = \frac{m_t}{1 - \beta_1^t} \qquad \hat{v}_t = \frac{v_t}{1 - \beta_2^t}$$

**Actualización**:
$$\theta_t = \theta_{t-1} - \eta \cdot \frac{\hat{m}_t}{\sqrt{\hat{v}_t} + \epsilon}$$

Valores típicos: $\beta_1 = 0.9$, $\beta_2 = 0.999$, $\epsilon = 10^{-8}$.

### ¿Qué hace la normalización?

La clave está en dividir por $\sqrt{\hat{v}_t}$. Si un parámetro tiene gradientes consistentemente grandes (digamos, siempre ~10.0), $\sqrt{\hat{v}_t} \approx 10$, entonces el paso efectivo es $\frac{\hat{m}_t}{10}$. Si otro parámetro tiene gradientes chiquitos (~0.01), $\sqrt{\hat{v}_t} \approx 0.01$, y el paso efectivo es $\frac{\hat{m}_t}{0.01}$, ¡mucho más grande!

**Es como si Adam "normalizara" los gradientes para que todos los parámetros avancen a un paso comparable.** Los parámetros con gradientes grandes dan pasos proporcionalmente chicos, y los de gradientes chicos dan pasos proporcionalmente grandes.

### Ejemplo numérico: por qué importa

Supongamos que tenemos 3 parámetros con gradientes muy diferentes:

In [ ]:
# Numerical example: fixed LR vs adaptive (Adam-like) updates

gradients = np.array([10.0, 0.01, -3.0])
lr = 0.01

print("=== 3 parámetros con gradientes MUY diferentes ===")
print(f"Gradientes: {gradients}")
print(f"Learning rate: {lr}")
print()

# Fixed LR update
update_fixed = lr * gradients
print("--- Con LR fijo (SGD) ---")
print(f"Actualización = LR × gradiente")
for i, (g, u) in enumerate(zip(gradients, update_fixed)):
    print(f"  Param {i+1}: {lr} × {g:>7.2f} = {u:>8.4f}")
print(f"  → Param 1 se mueve 1000× más que Param 2!")
print(f"  → Si ajusto el LR para Param 1, Param 2 no aprende")
print(f"  → Si ajusto el LR para Param 2, Param 1 explota")

print()

# Adaptive update (Adam-like: normalize by gradient magnitude)
# Simulating after a few steps where v has accumulated
v_estimate = gradients ** 2  # simplified: assume v ≈ g²
update_adaptive = lr * gradients / (np.sqrt(v_estimate) + 1e-8)

print("--- Con Adam (adaptativo) ---")
print(f"Actualización = LR × gradiente / √(v + ε)")
for i, (g, v, u) in enumerate(zip(gradients, v_estimate, update_adaptive)):
    print(f"  Param {i+1}: {lr} × {g:>7.2f} / √{v:>8.2f} = {u:>8.4f}")
print(f"  → ¡Todos los parámetros se mueven ±{lr}!")
print(f"  → Cada uno avanza a su propio ritmo, normalizado")
print(f"  → No importa la escala del gradiente")

print()
print("=" * 60)
print("La normalización de Adam hace que el paso efectivo sea")
print("similar para todos los parámetros, independientemente")
print("de la magnitud de sus gradientes.")
print("\nEsto es especialmente útil en redes profundas donde")
print("los gradientes pueden variar en ÓRDENES DE MAGNITUD")
print("entre diferentes capas.")

In [ ]:
# Full Adam implementation from scratch + comparison
np.random.seed(42)

class AdamOptimizer:
    """Adam optimizer from scratch."""
    def __init__(self, lr=0.01, beta1=0.9, beta2=0.999, eps=1e-8):
        self.lr = lr
        self.beta1 = beta1
        self.beta2 = beta2
        self.eps = eps
        self.m = None  # first moment
        self.v = None  # second moment
        self.t = 0     # timestep
    
    def step(self, params, grads):
        if self.m is None:
            self.m = np.zeros_like(params)
            self.v = np.zeros_like(params)
        
        self.t += 1
        
        # Update biased moments
        self.m = self.beta1 * self.m + (1 - self.beta1) * grads
        self.v = self.beta2 * self.v + (1 - self.beta2) * grads**2
        
        # Bias correction
        m_hat = self.m / (1 - self.beta1**self.t)
        v_hat = self.v / (1 - self.beta2**self.t)
        
        # Update
        params -= self.lr * m_hat / (np.sqrt(v_hat) + self.eps)
        return params


# Extremely elongated loss surface (100x difference in curvature)
def elongated_loss(params):
    x, y = params
    return 0.01 * (x - 5)**2 + 100 * (y - 2)**2

def elongated_grad(params):
    x, y = params
    return np.array([0.02 * (x - 5), 200 * (y - 2)])

# Optimize with different methods
n_steps = 200

# SGD
params_sgd = np.array([-3.0, 5.0])
path_sgd = [params_sgd.copy()]
for _ in range(n_steps):
    g = elongated_grad(params_sgd)
    params_sgd -= 0.003 * g  # small LR needed for stability
    path_sgd.append(params_sgd.copy())

# SGD + Momentum
params_mom = np.array([-3.0, 5.0])
path_mom = [params_mom.copy()]
vel = np.zeros(2)
for _ in range(n_steps):
    g = elongated_grad(params_mom)
    vel = 0.9 * vel + 0.1 * g
    params_mom -= 0.003 * vel
    path_mom.append(params_mom.copy())

# Adam
params_adam = np.array([-3.0, 5.0])
path_adam = [params_adam.copy()]
adam = AdamOptimizer(lr=0.3)
for _ in range(n_steps):
    g = elongated_grad(params_adam)
    params_adam = adam.step(params_adam, g)
    path_adam.append(params_adam.copy())

# Plot
path_sgd = np.array(path_sgd)
path_mom = np.array(path_mom)
path_adam = np.array(path_adam)

x_r = np.linspace(-5, 10, 200)
y_r = np.linspace(-1, 6, 200)
X, Y = np.meshgrid(x_r, y_r)
Z = 0.01 * (X - 5)**2 + 100 * (Y - 2)**2

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

ax = axes[0]
ax.contour(X, Y, Z, levels=np.logspace(-1, 4, 30), cmap='viridis', alpha=0.5)
ax.plot(path_sgd[:, 0], path_sgd[:, 1], 'r.-', markersize=3, linewidth=1,
        alpha=0.7, label='SGD')
ax.plot(path_mom[:, 0], path_mom[:, 1], 'b.-', markersize=3, linewidth=1,
        alpha=0.7, label='SGD + Momentum')
ax.plot(path_adam[:, 0], path_adam[:, 1], 'g.-', markersize=3, linewidth=1.5,
        alpha=0.9, label='Adam')
ax.plot(5, 2, 'k*', markersize=15, label='Mínimo')
ax.set_xlabel('θ₁', fontsize=12)
ax.set_ylabel('θ₂', fontsize=12)
ax.set_title('Trayectorias: SGD vs Momentum vs Adam', fontsize=12)
ax.legend(fontsize=10)

ax = axes[1]
losses_sgd = [elongated_loss(p) for p in path_sgd]
losses_mom = [elongated_loss(p) for p in path_mom]
losses_adam = [elongated_loss(p) for p in path_adam]
ax.plot(losses_sgd, 'r-', alpha=0.7, label='SGD')
ax.plot(losses_mom, 'b-', alpha=0.7, label='SGD + Momentum')
ax.plot(losses_adam, 'g-', alpha=0.7, linewidth=2, label='Adam')
ax.set_xlabel('Paso', fontsize=11)
ax.set_ylabel('Loss', fontsize=11)
ax.set_yscale('log')
ax.set_title('Convergencia: SGD vs Momentum vs Adam', fontsize=12)
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"Loss final SGD:      {losses_sgd[-1]:.6f}")
print(f"Loss final Momentum: {losses_mom[-1]:.6f}")
print(f"Loss final Adam:     {losses_adam[-1]:.6f}")

Adam es claramente superior en esta superficie extremadamente alargada. La normalización adaptativa hace que el paso sea proporcionado en ambas dimensiones, evitando tanto el zigzagueo como la convergencia lenta.

### ¿Por qué funciona la corrección de sesgo?

Los momentos $m$ y $v$ empiezan en 0. En los primeros pasos, están subestimados porque todavía no acumularon suficiente historia. La corrección de sesgo ($\frac{m}{1-\beta^t}$) compensa esto, "inflando" los momentos al principio. A medida que $t$ crece, $\beta^t \to 0$ y la corrección desaparece.

### Variantes de Adam

- **AdamW**: Adam con weight decay decoupled. En vez de agregar el peso a la loss (L2), lo resta directamente del parámetro. Es la versión estándar hoy.
- **RAdam**: agrega un warm-up automático basado en la varianza.
- **LAMB/LARS**: para batch sizes extremadamente grandes.

---

<a id='inicializacion'></a>
## 7. Inicialización de parámetros

Antes de hacer el primer paso de gradient descent, necesitamos darle valores iniciales a todos los parámetros. Esto parece un detalle menor, pero una mala inicialización puede hacer que el entrenamiento **no funcione directamente**.

### ¿Por qué no inicializar todo en cero?

Parece lo más simple: todos los weights en 0, todos los biases en 0. Pero tiene un problema fatal: **rompe la simetría**.

Si todos los weights de una capa son iguales, todas las neuronas de esa capa computan exactamente lo mismo. En el forward pass, producen el mismo output. En el backward pass, reciben el mismo gradiente. Se actualizan igual. **Son clones.**

¿Para qué tener 256 neuronas si todas hacen lo mismo? Es como tener una sola neurona multiplicada por 256. El modelo pierde toda su capacidad expresiva.

### La solución: inicialización aleatoria (pero controlada)

Inicializamos con valores aleatorios para romper la simetría. Pero la **escala** importa mucho:

- **Valores muy grandes**: las preactivaciones ($z = Wx + b$) son grandes → las activaciones se saturan (sigmoid/tanh en los extremos, ReLU siempre activa o muerta) → gradientes desaparecen o explotan.
- **Valores muy chicos**: las preactivaciones son chiquitas → la señal se desvanece a lo largo de las capas → gradientes diminutos → no aprende nada.

Lo ideal es que las preactivaciones mantengan una **varianza estable** a lo largo de las capas, tanto en forward como en backward.

### Fan-in scaling

Consideremos una neurona con $n_{in}$ inputs (el "fan-in" de la capa). Su preactivación es:

$$z = \sum_{j=1}^{n_{in}} w_j x_j$$

Si $x_j$ y $w_j$ son independientes con media 0:

$$\text{Var}(z) = n_{in} \cdot \text{Var}(w) \cdot \text{Var}(x)$$

Para que $\text{Var}(z) = \text{Var}(x)$ (que la varianza se preserve), necesitamos:

$$\text{Var}(w) = \frac{1}{n_{in}}$$

### Xavier/Glorot Init

**Xavier init** (Glorot & Bengio, 2010) busca preservar la varianza tanto en forward como en backward. Promedia fan-in y fan-out:

$$W \sim \mathcal{N}\left(0, \frac{2}{n_{in} + n_{out}}\right) \quad \text{o} \quad W \sim \text{Uniform}\left(-\sqrt{\frac{6}{n_{in} + n_{out}}}, \sqrt{\frac{6}{n_{in} + n_{out}}}\right)$$

Funciona bien para **tanh y sigmoid** (activaciones que preservan aproximadamente la varianza cerca de 0).

### He/Kaiming Init (para ReLU)

ReLU mata la mitad de las activaciones (las negativas → 0). Esto reduce la varianza a la mitad. Para compensar, **Kaiming init** (He et al., 2015) duplica la varianza:

$$W \sim \mathcal{N}\left(0, \frac{2}{n_{in}}\right)$$

El factor 2 es el **gain** de ReLU ($\sqrt{2}$ en desviación estándar). Para Leaky ReLU, el gain es $\sqrt{\frac{2}{1 + \alpha^2}}$ donde $\alpha$ es el slope negativo.

### Inicialización de la última capa

Un truco práctico: la última capa (que produce logits o la predicción final) se suele inicializar con **valores muy chicos** (o directamente en cero). ¿Por qué?

- Al inicio del entrenamiento, queremos que el modelo sea "inseguro": que le asigne probabilidad similar a todas las clases.
- Si los logits iniciales son grandes, el modelo empieza con predicciones muy confiadas pero incorrectas, y la loss es enorme.
- Con logits chicos (cercanos a 0), softmax produce una distribución casi uniforme, la loss inicial es $-\log(1/C) = \log(C)$ (donde C es el número de clases), que es un punto de partida razonable.

In [ ]:
# Demonstrate the effect of initialization on preactivation distributions
np.random.seed(42)

# Simulate a 5-layer network with ReLU
input_dim = 256
hidden_dim = 256
n_layers = 5
n_samples = 1000

# Generate random input
x = np.random.randn(n_samples, input_dim)

def forward_pass(x, init_type, n_layers=5, dim=256):
    """Forward pass through multiple layers with ReLU, tracking preactivation stats."""
    activations = [x]
    preact_stds = [x.std()]
    h = x
    
    for layer in range(n_layers):
        # Initialize weights based on type
        if init_type == 'too_small':
            W = np.random.randn(dim, dim) * 0.01
        elif init_type == 'too_large':
            W = np.random.randn(dim, dim) * 1.0
        elif init_type == 'xavier':
            W = np.random.randn(dim, dim) * np.sqrt(2.0 / (dim + dim))
        elif init_type == 'kaiming':
            W = np.random.randn(dim, dim) * np.sqrt(2.0 / dim)
        
        # Preactivation
        z = h @ W
        preact_stds.append(z.std())
        
        # ReLU activation
        h = np.maximum(0, z)
        activations.append(h)
    
    return activations, preact_stds

# Run all init types
init_types = ['too_small', 'too_large', 'xavier', 'kaiming']
labels = ['Muy chica (σ=0.01)', 'Muy grande (σ=1.0)', 'Xavier', 'Kaiming (He)']
colors = ['red', 'orange', 'blue', 'green']

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Top row: preactivation std across layers
ax = axes[0, 0]
for init_type, label, color in zip(init_types, labels, colors):
    _, stds = forward_pass(x, init_type)
    ax.plot(range(len(stds)), stds, 'o-', label=label, color=color, linewidth=2)
ax.set_xlabel('Capa', fontsize=11)
ax.set_ylabel('Std de preactivaciones', fontsize=11)
ax.set_title('Desviación estándar de preactivaciones por capa', fontsize=12)
ax.set_yscale('log')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)

# Top right: histogram of final layer activations
ax = axes[0, 1]
for init_type, label, color in zip(init_types, labels, colors):
    acts, _ = forward_pass(x, init_type)
    final_act = acts[-1].flatten()
    # Clip for visualization
    final_act = np.clip(final_act, -10, 10)
    ax.hist(final_act, bins=50, alpha=0.5, label=label, color=color, density=True)
ax.set_xlabel('Valor de activación', fontsize=11)
ax.set_ylabel('Densidad', fontsize=11)
ax.set_title('Distribución de activaciones en la última capa', fontsize=12)
ax.legend(fontsize=9)
ax.set_xlim(-5, 10)

# Bottom: detailed view of Xavier vs Kaiming per layer
for idx, (init_type, label) in enumerate([('xavier', 'Xavier'), ('kaiming', 'Kaiming')]):
    ax = axes[1, idx]
    acts, _ = forward_pass(x, init_type)
    for i, act in enumerate(acts):
        vals = act.flatten()
        vals = vals[vals != 0]  # remove dead neurons for visibility
        if len(vals) > 0:
            ax.hist(vals, bins=50, alpha=0.4, label=f'Capa {i}', density=True)
    ax.set_xlabel('Valor de activación', fontsize=11)
    ax.set_ylabel('Densidad', fontsize=11)
    ax.set_title(f'Activaciones por capa ({label} init + ReLU)', fontsize=12)
    ax.legend(fontsize=8)
    ax.set_xlim(-1, 8)

plt.tight_layout()
plt.show()

print("Con inicialización muy chica: la señal desaparece (stds → 0)")
print("Con inicialización muy grande: la señal explota (stds → ∞)")
print("Xavier: buena, pero pierde varianza con ReLU (no compensa el factor 1/2)")
print("Kaiming: PERFECTA para ReLU — la varianza se mantiene estable")

In [ ]:
# PyTorch: verify initializations are built-in

# Create a layer
layer = nn.Linear(256, 256)

# Default PyTorch init (Kaiming uniform)
print("=== Inicializaciones en PyTorch ===")
print(f"Default (Kaiming uniform):")
print(f"  Weight std: {layer.weight.data.std():.4f}")
print(f"  Expected:   {np.sqrt(2.0 / 256):.4f} (Kaiming) or {1/np.sqrt(256):.4f} (fan-in)")
print()

# Xavier init
nn.init.xavier_normal_(layer.weight)
print(f"Xavier normal:")
print(f"  Weight std: {layer.weight.data.std():.4f}")
print(f"  Expected:   {np.sqrt(2.0 / (256 + 256)):.4f}")
print()

# Kaiming (He) init
nn.init.kaiming_normal_(layer.weight, mode='fan_in', nonlinearity='relu')
print(f"Kaiming normal (for ReLU):")
print(f"  Weight std: {layer.weight.data.std():.4f}")
print(f"  Expected:   {np.sqrt(2.0 / 256):.4f}")
print()

# Small init for output layer
output_layer = nn.Linear(256, 10)  # 10 classes
nn.init.normal_(output_layer.weight, mean=0, std=0.01)
nn.init.zeros_(output_layer.bias)
print(f"Output layer (small init):")
print(f"  Weight std: {output_layer.weight.data.std():.4f}")
print(f"  Bias: {output_layer.bias.data}")
print(f"  Logits iniciales serán ~0 → softmax ≈ uniforme → loss ≈ log(10) = {np.log(10):.4f}")

### Resumen de inicialización

| Activación | Init recomendada | Varianza | Gain |
|:-----------|:-----------------|:---------|:-----|
| Linear/Tanh/Sigmoid | Xavier | $\frac{2}{n_{in} + n_{out}}$ | 1 |
| ReLU | Kaiming (He) | $\frac{2}{n_{in}}$ | $\sqrt{2}$ |
| Leaky ReLU (α) | Kaiming | $\frac{2}{(1+\alpha^2) \cdot n_{in}}$ | $\sqrt{\frac{2}{1+\alpha^2}}$ |
| Última capa | Small/Zero | ~0.01 o 0 | — |

La regla es simple: **Kaiming para ReLU, Xavier para el resto, chiquita para la última capa.** PyTorch ya usa Kaiming por default, así que en la práctica casi nunca tenés que cambiar nada.

---

<a id='batchnorm'></a>
## 8. BatchNorm (Batch Normalization)

Aún con buena inicialización, a medida que la red aprende y los parámetros cambian, la distribución de las preactivaciones en cada capa va cambiando. Esto se llama **internal covariate shift**: cada capa tiene que adaptarse constantemente a distribuciones cambiantes de su input.

**BatchNorm** (Ioffe & Szegedy, 2015) resuelve esto normalizando las preactivaciones **en cada paso de entrenamiento**.

### ¿Cómo funciona?

Para cada neurona (cada feature), BatchNorm calcula la media y varianza **a través del batch**, y normaliza:

**Paso 1: Calcular estadísticas del batch**
$$\mu_B = \frac{1}{|B|} \sum_{i \in B} z_i \qquad \sigma_B^2 = \frac{1}{|B|} \sum_{i \in B} (z_i - \mu_B)^2$$

Donde $z_i$ es la preactivación de la neurona para el ejemplo $i$ del batch.

**Paso 2: Normalizar**
$$\hat{z}_i = \frac{z_i - \mu_B}{\sqrt{\sigma_B^2 + \epsilon}}$$

Ahora $\hat{z}$ tiene media 0 y varianza 1. Pero ojo: no siempre queremos media 0 y varianza 1 (la red puede necesitar otro rango). Entonces...

**Paso 3: Scale and shift (parámetros aprendibles)**
$$y_i = \gamma \cdot \hat{z}_i + \beta$$

Donde $\gamma$ y $\beta$ son **parámetros aprendibles** (uno por neurona). Empiezan en $\gamma = 1$, $\beta = 0$ (identidad), y la red aprende qué distribución le conviene.

### ¿Por qué necesitamos γ y β?

Si solo normalizamos, estamos forzando a todas las neuronas a tener media 0 y varianza 1. Pero capaz una neurona funciona mejor con media 3 y varianza 2. Los parámetros $\gamma$ y $\beta$ le dan flexibilidad: la red puede "deshacerse" de la normalización si le conviene.

Es como decir: "empezamos normalizados (que es un buen punto de partida), y dejamos que la red ajuste a partir de ahí".

### Inferencia: el problema del batch

En entrenamiento, calculamos $\mu_B$ y $\sigma_B^2$ usando el batch actual. Pero en inferencia, podríamos tener un solo ejemplo. ¿Qué media y varianza usamos?

La solución: durante el entrenamiento, mantenemos un **running average** de la media y varianza:

$$\mu_{\text{running}} = (1 - \alpha) \cdot \mu_{\text{running}} + \alpha \cdot \mu_B$$
$$\sigma^2_{\text{running}} = (1 - \alpha) \cdot \sigma^2_{\text{running}} + \alpha \cdot \sigma^2_B$$

Con $\alpha$ típicamente 0.1. En inferencia, usamos estas estadísticas acumuladas en vez de las del batch.

### ¿Dónde se pone BatchNorm?

Lo más común es **después de la operación lineal y antes de la activación**:

```
z = W @ x + b     ← operación lineal
z_bn = BatchNorm(z)  ← normalizar
h = ReLU(z_bn)       ← activación
```

Nota: cuando se usa BatchNorm, el bias $b$ de la capa lineal es redundante (BatchNorm resta la media, así que el bias desaparece). Por eso en PyTorch se suele usar `nn.Linear(in, out, bias=False)` antes de BatchNorm.

In [ ]:
# BatchNorm from scratch
np.random.seed(42)

class BatchNorm1d:
    """BatchNorm implementation from scratch."""
    def __init__(self, num_features, eps=1e-5, momentum=0.1):
        self.eps = eps
        self.momentum = momentum
        
        # Learnable parameters
        self.gamma = np.ones(num_features)   # scale
        self.beta = np.zeros(num_features)   # shift
        
        # Running statistics (for inference)
        self.running_mean = np.zeros(num_features)
        self.running_var = np.ones(num_features)
        
        self.training = True
    
    def forward(self, z):
        """
        z: (batch_size, num_features)
        """
        if self.training:
            # Compute batch statistics
            batch_mean = z.mean(axis=0)         # (num_features,)
            batch_var = z.var(axis=0)            # (num_features,)
            
            # Normalize
            z_hat = (z - batch_mean) / np.sqrt(batch_var + self.eps)
            
            # Update running stats
            self.running_mean = (1 - self.momentum) * self.running_mean + self.momentum * batch_mean
            self.running_var = (1 - self.momentum) * self.running_var + self.momentum * batch_var
        else:
            # Use running stats for inference
            z_hat = (z - self.running_mean) / np.sqrt(self.running_var + self.eps)
        
        # Scale and shift
        out = self.gamma * z_hat + self.beta
        return out


# Demo: show how BatchNorm stabilizes preactivations
batch_size = 64
num_features = 128

# Simulate preactivations with varying statistics (like during training)
fig, axes = plt.subplots(2, 3, figsize=(15, 8))

bn = BatchNorm1d(num_features)

for i in range(3):
    # Simulate preactivations with different means/variances
    # (as if the network parameters are changing during training)
    mean_shift = np.random.randn(num_features) * (i + 1) * 2
    var_scale = np.abs(np.random.randn(num_features)) * (i + 1) + 0.5
    z = np.random.randn(batch_size, num_features) * var_scale + mean_shift
    
    # Before BatchNorm
    axes[0, i].hist(z.flatten(), bins=50, alpha=0.7, color='red', density=True)
    axes[0, i].set_title(f'Paso {i+1}: ANTES de BN\nμ≈{z.mean():.1f}, σ≈{z.std():.1f}', fontsize=10)
    axes[0, i].set_xlim(-15, 15)
    
    # After BatchNorm
    z_bn = bn.forward(z)
    axes[1, i].hist(z_bn.flatten(), bins=50, alpha=0.7, color='blue', density=True)
    axes[1, i].set_title(f'Paso {i+1}: DESPUÉS de BN\nμ≈{z_bn.mean():.2f}, σ≈{z_bn.std():.2f}', fontsize=10)
    axes[1, i].set_xlim(-5, 5)

axes[0, 0].set_ylabel('Sin BatchNorm', fontsize=12, color='red')
axes[1, 0].set_ylabel('Con BatchNorm', fontsize=12, color='blue')

plt.suptitle('BatchNorm estabiliza la distribución de preactivaciones', fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

print("Sin BatchNorm: la distribución cambia salvajemente entre pasos")
print("Con BatchNorm: siempre centrada y normalizada → entrenamiento más estable")

In [ ]:
# PyTorch: BatchNorm in practice

# Simple network WITH and WITHOUT BatchNorm
class NetNoBN(nn.Module):
    def __init__(self):
        super().__init__()
        self.layers = nn.Sequential(
            nn.Linear(10, 64),
            nn.ReLU(),
            nn.Linear(64, 64),
            nn.ReLU(),
            nn.Linear(64, 64),
            nn.ReLU(),
            nn.Linear(64, 64),
            nn.ReLU(),
            nn.Linear(64, 2),
        )
    def forward(self, x):
        return self.layers(x)

class NetWithBN(nn.Module):
    def __init__(self):
        super().__init__()
        self.layers = nn.Sequential(
            nn.Linear(10, 64, bias=False),  # bias=False because BN handles it
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.Linear(64, 64, bias=False),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.Linear(64, 64, bias=False),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.Linear(64, 64, bias=False),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.Linear(64, 2),
        )
    def forward(self, x):
        return self.layers(x)

# Generate synthetic classification data
torch.manual_seed(42)
X_train = torch.randn(1000, 10)
y_train = (X_train[:, 0] + X_train[:, 1] > 0).long()

# Train both
def train_model(model, X, y, lr=0.01, epochs=200, batch_size=64):
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    criterion = nn.CrossEntropyLoss()
    losses = []
    
    for epoch in range(epochs):
        epoch_loss = 0
        n_batches = 0
        # Shuffle
        perm = torch.randperm(len(X))
        for i in range(0, len(X), batch_size):
            idx = perm[i:i+batch_size]
            xb, yb = X[idx], y[idx]
            
            pred = model(xb)
            loss = criterion(pred, yb)
            
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            
            epoch_loss += loss.item()
            n_batches += 1
        
        losses.append(epoch_loss / n_batches)
    
    return losses

model_no_bn = NetNoBN()
model_bn = NetWithBN()

losses_no_bn = train_model(model_no_bn, X_train, y_train)
losses_bn = train_model(model_bn, X_train, y_train)

plt.figure(figsize=(10, 5))
plt.plot(losses_no_bn, 'r-', label='Sin BatchNorm', alpha=0.7)
plt.plot(losses_bn, 'b-', label='Con BatchNorm', alpha=0.7, linewidth=2)
plt.xlabel('Epoch', fontsize=11)
plt.ylabel('Loss', fontsize=11)
plt.title('Entrenamiento con y sin BatchNorm', fontsize=13)
plt.legend(fontsize=11)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f"Loss final sin BN: {losses_no_bn[-1]:.4f}")
print(f"Loss final con BN: {losses_bn[-1]:.4f}")

### Ventajas de BatchNorm

1. **Estabiliza el entrenamiento**: las preactivaciones están siempre en un rango controlado
2. **Permite learning rates más grandes**: como la distribución está normalizada, el optimizador puede ser más agresivo
3. **Regulariza ligeramente**: el ruido de las estadísticas del batch actúa como regularización
4. **Menos sensible a la inicialización**: como normaliza en cada paso, la inicialización importa menos

### Desventajas

1. **Depende del batch size**: con batches chicos, las estadísticas son ruidosas y el entrenamiento se degrada
2. **Comportamiento diferente en train vs eval**: hay que recordar llamar `model.eval()` en inferencia
3. **No funciona bien con batch size 1**: no podés calcular una media/varianza de un solo ejemplo
4. **Problemático para modelos autoregresivos**: donde el batch puede tener secuencias de diferente largo

Para estos casos, existe **LayerNorm**.

---

<a id='layernorm'></a>
## 9. LayerNorm (Layer Normalization)

LayerNorm es la alternativa a BatchNorm que **no depende del batch**. En vez de normalizar a través de los ejemplos del batch (para cada feature), normaliza **a través de los features** (para cada ejemplo).

### La diferencia clave

```
BatchNorm: normaliza por feature, A TRAVÉS del batch
           → necesita batch > 1, estadísticas del batch

Input:  (batch_size, features)
         ↓↓↓↓↓↓↓↓↓↓           ← normaliza en esta dirección (por columna)
         [x₁₁  x₁₂  x₁₃]
         [x₂₁  x₂₂  x₂₃]     media y std por feature (columna)
         [x₃₁  x₃₂  x₃₃]
         [x₄₁  x₄₂  x₄₃]

LayerNorm: normaliza por ejemplo, A TRAVÉS de los features
           → funciona con batch size 1, independiente del batch

Input:  (batch_size, features)
         [x₁₁  x₁₂  x₁₃] →→→  ← normaliza en esta dirección (por fila)
         [x₂₁  x₂₂  x₂₃] →→→  media y std por ejemplo (fila)
         [x₃₁  x₃₂  x₃₃] →→→
         [x₄₁  x₄₂  x₄₃] →→→
```

### La fórmula

Para cada ejemplo $i$:

$$\mu_i = \frac{1}{D} \sum_{j=1}^{D} z_{ij} \qquad \sigma_i^2 = \frac{1}{D} \sum_{j=1}^{D} (z_{ij} - \mu_i)^2$$

$$\hat{z}_{ij} = \frac{z_{ij} - \mu_i}{\sqrt{\sigma_i^2 + \epsilon}}$$

$$y_{ij} = \gamma_j \cdot \hat{z}_{ij} + \beta_j$$

Donde $D$ es el número de features.

### ¿Cuándo usar LayerNorm vs BatchNorm?

| Situación | Usar |
|:----------|:-----|
| CNNs con batches grandes | BatchNorm |
| Transformers / modelos de lenguaje | **LayerNorm** |
| Batch size = 1 | **LayerNorm** |
| Modelos autoregresivos | **LayerNorm** |
| RNNs | **LayerNorm** |

LayerNorm es especialmente popular en **Transformers** (GPT, BERT, etc.) porque:
- No depende del batch → funciona igual en train y eval
- Funciona con secuencias de largo variable
- Es más simple (no necesita running statistics)

In [ ]:
# Compare BatchNorm vs LayerNorm visually
np.random.seed(42)

batch_size = 4
features = 5

# Random preactivations
z = np.array([
    [10.0,  0.5,  -2.0,  3.0,  7.0],   # example 1
    [ 1.0,  0.1,  -0.3,  0.5,  0.8],   # example 2 (small values)
    [50.0, 20.0,  30.0, 40.0, 10.0],   # example 3 (large values)
    [-1.0, -2.0,  -3.0, -4.0, -5.0],   # example 4 (all negative)
])

print("=== Preactivaciones originales ===")
print(f"z =\n{z}")
print()

# BatchNorm: normalize per feature (across batch)
bn_mean = z.mean(axis=0)  # (features,)
bn_var = z.var(axis=0)    # (features,)
z_bn = (z - bn_mean) / np.sqrt(bn_var + 1e-5)

print("=== BatchNorm (normalizar por feature, a través del batch) ===")
print(f"Media por feature: {bn_mean}")
print(f"z_bn =\n{np.round(z_bn, 3)}")
print(f"Verificación: media por columna ≈ 0: {np.round(z_bn.mean(axis=0), 6)}")
print(f"Verificación: std por columna ≈ 1:   {np.round(z_bn.std(axis=0), 3)}")
print()

# LayerNorm: normalize per example (across features)
ln_mean = z.mean(axis=1, keepdims=True)  # (batch, 1)
ln_var = z.var(axis=1, keepdims=True)    # (batch, 1)
z_ln = (z - ln_mean) / np.sqrt(ln_var + 1e-5)

print("=== LayerNorm (normalizar por ejemplo, a través de features) ===")
print(f"Media por ejemplo: {ln_mean.flatten()}")
print(f"z_ln =\n{np.round(z_ln, 3)}")
print(f"Verificación: media por fila ≈ 0: {np.round(z_ln.mean(axis=1), 6)}")
print(f"Verificación: std por fila ≈ 1:   {np.round(z_ln.std(axis=1), 3)}")

In [ ]:
# Visual comparison
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# Original
im0 = axes[0].imshow(z, cmap='RdBu_r', aspect='auto', vmin=-50, vmax=50)
axes[0].set_title('Original', fontsize=12)
axes[0].set_xlabel('Features')
axes[0].set_ylabel('Ejemplos del batch')
for i in range(batch_size):
    for j in range(features):
        axes[0].text(j, i, f'{z[i,j]:.1f}', ha='center', va='center', fontsize=9)
plt.colorbar(im0, ax=axes[0])

# BatchNorm
im1 = axes[1].imshow(z_bn, cmap='RdBu_r', aspect='auto', vmin=-2, vmax=2)
axes[1].set_title('BatchNorm\n(normaliza ↓ por columna)', fontsize=12)
axes[1].set_xlabel('Features')
axes[1].set_ylabel('Ejemplos del batch')
for i in range(batch_size):
    for j in range(features):
        axes[1].text(j, i, f'{z_bn[i,j]:.2f}', ha='center', va='center', fontsize=8)
plt.colorbar(im1, ax=axes[1])

# LayerNorm
im2 = axes[2].imshow(z_ln, cmap='RdBu_r', aspect='auto', vmin=-2, vmax=2)
axes[2].set_title('LayerNorm\n(normaliza → por fila)', fontsize=12)
axes[2].set_xlabel('Features')
axes[2].set_ylabel('Ejemplos del batch')
for i in range(batch_size):
    for j in range(features):
        axes[2].text(j, i, f'{z_ln[i,j]:.2f}', ha='center', va='center', fontsize=8)
plt.colorbar(im2, ax=axes[2])

plt.tight_layout()
plt.show()

print("BatchNorm: cada COLUMNA (feature) tiene media≈0, std≈1")
print("LayerNorm: cada FILA (ejemplo) tiene media≈0, std≈1")

La diferencia visual es clara: BatchNorm normaliza cada columna (feature) y LayerNorm normaliza cada fila (ejemplo). Ambas logran estabilizar las distribuciones, pero LayerNorm no depende del batch, lo que la hace ideal para Transformers y modelos donde el batch size puede variar.

---

<a id='residual'></a>
## 10. Residual connections (skip connections)

Las residual connections son quizás la innovación arquitectónica más importante de deep learning después de la convolución. Fueron introducidas por He et al. en 2015 (ResNet) y **cambiaron completamente lo que era posible entrenar**.

### El problema: degradación con profundidad

En teoría, una red más profunda debería ser al menos tan buena como una más superficial (la red profunda puede aprender a "no hacer nada" en las capas extra). Pero en la práctica, agregar capas a una red **empeoraba** el rendimiento:

- **Vanishing gradients**: en el backward pass, los gradientes se multiplican capa por capa. Si cada multiplicación reduce el gradiente un poco, después de 50+ capas el gradiente es prácticamente cero. Las primeras capas no aprenden nada.
- **Degradación del entrenamiento**: ni siquiera la loss de entrenamiento mejoraba con más capas. No era un problema de overfitting — era un problema de **optimización**.

### La idea: "branch" que suma el input al output

Una residual connection es brutalmente simple: en vez de aprender $h(x)$ directamente, la capa aprende el **residuo** $f(x) = h(x) - x$, y el output final es:

$$\text{output} = f(x) + x$$

Gráficamente:

```
x ─────────────────────────── (+) → output
│                              ↑
└──→ [capa/bloque] → f(x) ────┘
```

La rama de abajo es el "branch" que aprende. La rama de arriba es el "skip connection" que simplemente pasa $x$ directo.

![Residual connections (skip connections)](../ai_notas/AI%20notas/image%2049.png)

![Diagrama de un bloque residual](../ai_notas/AI%20notas/image%2050.png)

### ¿Por qué funciona tan bien?

**1. Camino directo para los gradientes:**

En el backward pass, el gradiente fluye por la suma:

$$\frac{\partial \text{output}}{\partial x} = \frac{\partial f(x)}{\partial x} + 1$$

Ese $+1$ es la clave. Sin importar qué tan chico sea $\frac{\partial f(x)}{\partial x}$, el gradiente siempre es **al menos 1**. Los gradientes ya no se desvanecen porque tienen un "atajo" directo.

**2. Aprender la identidad es fácil:**

Si una capa no necesita transformar el input (solo necesita pasarlo), la rama $f(x)$ simplemente aprende a dar 0 (todos los weights → 0). Esto es mucho más fácil que aprender la función identidad explícitamente.

**3. Permite redes absurdamente profundas:**

Antes de ResNet, las redes prácticas tenían ~20 capas. ResNet demostró que se podían entrenar redes de **152 capas** sin degradación. Hoy en día, modelos como los Transformers usan cientos de capas con residual connections.

![ResNets permiten entrenar redes mucho más profundas](../ai_notas/AI%20notas/image%2051.png)

### Orden de operaciones (pre-activation vs post-activation)

Hay dos formas de organizar las operaciones dentro de un bloque residual:

**Post-activation (original ResNet):**
```
x → Linear → BN → ReLU → Linear → BN → (+x) → ReLU
```

**Pre-activation (ResNet v2, generalmente mejor):**
```
x → BN → ReLU → Linear → BN → ReLU → Linear → (+x)
```

En pre-activation, la activación va **antes** de la transformación lineal, y la suma con el skip connection va al final sin activación después. Esto permite un flujo de gradientes aún más limpio.

En Transformers, el orden típico es:
```
x → LayerNorm → Atención/FFN → (+x)
```

Que es esencialmente un bloque pre-activation con LayerNorm.

In [ ]:
# Demonstrate the effect of residual connections on training deep networks

class DeepNet(nn.Module):
    """Plain deep network (no skip connections)."""
    def __init__(self, depth=20, hidden=64):
        super().__init__()
        layers = [nn.Linear(10, hidden), nn.ReLU()]
        for _ in range(depth - 2):
            layers.extend([nn.Linear(hidden, hidden), nn.ReLU()])
        layers.append(nn.Linear(hidden, 2))
        self.net = nn.Sequential(*layers)
    
    def forward(self, x):
        return self.net(x)


class ResidualBlock(nn.Module):
    """A single residual block: LN → ReLU → Linear → LN → ReLU → Linear → (+x)."""
    def __init__(self, dim):
        super().__init__()
        self.ln1 = nn.LayerNorm(dim)
        self.linear1 = nn.Linear(dim, dim)
        self.ln2 = nn.LayerNorm(dim)
        self.linear2 = nn.Linear(dim, dim)
    
    def forward(self, x):
        # Pre-activation residual block
        h = self.ln1(x)
        h = torch.relu(h)
        h = self.linear1(h)
        h = self.ln2(h)
        h = torch.relu(h)
        h = self.linear2(h)
        return x + h  # ← residual connection!


class ResNet(nn.Module):
    """Deep network with residual connections."""
    def __init__(self, n_blocks=10, hidden=64):
        super().__init__()
        self.input_proj = nn.Linear(10, hidden)
        self.blocks = nn.ModuleList([ResidualBlock(hidden) for _ in range(n_blocks)])
        self.output = nn.Linear(hidden, 2)
    
    def forward(self, x):
        x = self.input_proj(x)
        for block in self.blocks:
            x = block(x)
        return self.output(x)


# Generate data
torch.manual_seed(42)
X = torch.randn(1000, 10)
y = (X[:, 0] * X[:, 1] + X[:, 2] > 0).long()  # non-linear decision boundary

# Train both: 20-layer plain vs 20-layer (10 blocks) residual
plain_net = DeepNet(depth=20, hidden=64)
res_net = ResNet(n_blocks=10, hidden=64)  # 10 blocks × 2 layers = 20 layers

print(f"Plain network parameters: {sum(p.numel() for p in plain_net.parameters()):,}")
print(f"ResNet parameters:        {sum(p.numel() for p in res_net.parameters()):,}")
print()

losses_plain = train_model(plain_net, X, y, lr=0.001, epochs=150)
losses_res = train_model(res_net, X, y, lr=0.001, epochs=150)

plt.figure(figsize=(10, 5))
plt.plot(losses_plain, 'r-', label='Plain 20-layer', alpha=0.7)
plt.plot(losses_res, 'b-', label='ResNet 20-layer (10 blocks)', alpha=0.7, linewidth=2)
plt.xlabel('Epoch', fontsize=11)
plt.ylabel('Loss', fontsize=11)
plt.title('Plain Deep Net vs ResNet (20 capas)', fontsize=13)
plt.legend(fontsize=11)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f"Loss final plain: {losses_plain[-1]:.4f}")
print(f"Loss final ResNet: {losses_res[-1]:.4f}")

In [ ]:
# Show gradient flow: how residual connections prevent vanishing gradients

def get_gradient_norms(model, X, y):
    """Get gradient norms for each layer's weights."""
    model.zero_grad()
    pred = model(X)
    loss = nn.CrossEntropyLoss()(pred, y)
    loss.backward()
    
    grad_norms = []
    for name, param in model.named_parameters():
        if 'weight' in name and param.grad is not None:
            grad_norms.append((name, param.grad.norm().item()))
    return grad_norms

# Get gradients for both
plain_net2 = DeepNet(depth=20, hidden=64)
res_net2 = ResNet(n_blocks=10, hidden=64)

grads_plain = get_gradient_norms(plain_net2, X[:64], y[:64])
grads_res = get_gradient_norms(res_net2, X[:64], y[:64])

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax = axes[0]
names, norms = zip(*grads_plain)
ax.bar(range(len(norms)), norms, color='red', alpha=0.7)
ax.set_xlabel('Capa (parámetros)', fontsize=10)
ax.set_ylabel('Norma del gradiente', fontsize=10)
ax.set_title('Plain network: gradientes\n(se desvanecen en capas tempranas)', fontsize=11)
ax.set_yscale('log')

ax = axes[1]
names, norms = zip(*grads_res)
ax.bar(range(len(norms)), norms, color='blue', alpha=0.7)
ax.set_xlabel('Capa (parámetros)', fontsize=10)
ax.set_ylabel('Norma del gradiente', fontsize=10)
ax.set_title('ResNet: gradientes\n(se mantienen gracias a skip connections)', fontsize=11)
ax.set_yscale('log')

plt.tight_layout()
plt.show()

print("En la plain network, los gradientes de las primeras capas son mucho")
print("más chicos que los de las últimas → vanishing gradient problem.")
print("\nEn ResNet, los gradientes se mantienen más uniformes gracias al")
print("camino directo del skip connection (el +1 en la derivada).")

![Regularización (tema del próximo notebook)](../ai_notas/AI%20notas/image%2048.png)

La imagen de arriba muestra conceptos de regularización que veremos en el próximo notebook. Por ahora, lo importante es entender que todas las técnicas de este notebook trabajan juntas para hacer posible el entrenamiento de redes profundas.

---

<a id='resumen'></a>
## 11. Resumen

| Concepto | Descripción |
|:---------|:------------|
| **Superficie no convexa** | La loss de redes neuronales tiene mínimos locales, saddle points y mesetas. Gradient descent puro se queda atascado |
| **SGD** | Usar minibatches aleatorios para calcular gradientes. El ruido ayuda a escapar mínimos malos y es más eficiente |
| **Minibatch / Epoch** | Minibatch: subconjunto sin reemplazo. Epoch: un pase completo. Cada batch define una loss function ligeramente diferente |
| **LR Schedule** | Empezar con LR grande (explorar) e ir achicando (converger). Cosine annealing y warmup+decay son los más usados |
| **Momentum** | Promedio exponencial de gradientes pasados. Cancela zigzagueo, acelera en direcciones consistentes. Típico: $\beta = 0.9$ |
| **Adam** | Momentum + normalización adaptativa por parámetro. Cada parámetro tiene su propio paso. El optimizer más usado |
| **Inicialización** | Kaiming (He) para ReLU ($\text{Var} = 2/n_{in}$), Xavier para tanh/sigmoid. Última capa con valores chicos |
| **BatchNorm** | Normaliza preactivaciones por feature a través del batch. Parámetros $\gamma, \beta$ aprendibles. Running stats para inferencia |
| **LayerNorm** | Normaliza por ejemplo a través de features. No depende del batch. Estándar en Transformers |
| **Residual connections** | Sumar el input al output: $y = f(x) + x$. Camino directo para gradientes ($+1$). Permite redes de 100+ capas |

### La combinación ganadora

La mayoría de los modelos modernos usan esta combinación:

1. **Adam** (o AdamW) como optimizer
2. **Cosine schedule** con warmup
3. **Kaiming init** para capas con ReLU
4. **LayerNorm** (en Transformers) o **BatchNorm** (en CNNs)
5. **Residual connections** en cada bloque

Con estas técnicas, podemos entrenar modelos con **miles de millones de parámetros** de forma estable. Sin ellas, ni siquiera podríamos entrenar una red de 50 capas.

---

**Siguiente notebook →** [08 - Regularización y Evaluación](./08_regularizacion_evaluacion.ipynb): overfitting, dropout, weight decay, early stopping, y cómo evaluar si tu modelo realmente funciona.